# Product Review Sentiment Analysis — Fine-Tuning

Fine-tunes `distilroberta-base` on the [Amazon Fine Food Reviews](https://www.kaggle.com/datasets/snap/amazon-fine-food-reviews) dataset (568,454 reviews) for 3-class sentiment classification (negative / neutral / positive), for use in a product review sentiment feature.

**Runtime:** Requires a GPU (Runtime → Change runtime type → T4 GPU or better).

**Input:** Upload `Reviews.csv` (the raw Amazon Fine Food Reviews CSV) to the Colab session's `/content/` directory before running.

**Output:** A fine-tuned model exported to a zip file, downloadable at the end — ready to drop into a serving layer (e.g. a FastAPI inference service).

**Final result from this run:** 82.8% accuracy, 0.827 macro-F1 on held-out test data (5 epochs, balanced 3-class dataset, ~128k rows).

## 1. Setup

Installs required packages. `torchvision` is explicitly uninstalled — it's pulled in as a side-effect dependency but has a known import bug (`VideoReader`) in some version combinations that breaks the `datasets` library's tensor formatting, even though this project never uses vision features at all.

**⚠️ After running this cell, restart the runtime** (Runtime → Restart session) before continuing to Section 2 — `torch`/`torchvision` are loaded into memory on import, so uninstalling alone doesn't undo that until the process restarts.

In [ ]:
!pip install -q transformers datasets accelerate scikit-learn
!pip uninstall -y torchvision -q

## 2. Imports and GPU check

Run this after restarting the runtime from Section 1.

In [ ]:
import re
import os
import json
import shutil
import inspect
import numpy as np
import pandas as pd
import torch
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score
from transformers import (
    AutoTokenizer, AutoModelForSequenceClassification,
    TrainingArguments, Trainer, DataCollatorWithPadding
)
from datasets import Dataset

print("GPU available:", torch.cuda.is_available())
assert torch.cuda.is_available(), "No GPU detected — check Runtime > Change runtime type"
if torch.cuda.is_available():
    print("Device:", torch.cuda.get_device_name(0))

## 3. Load and clean data

- Strips HTML artifacts (Amazon review text commonly contains `<br />` tags).
- Combines the `Summary` and `Text` columns into one `review_text` field, since both carry signal.
- Maps the 1-5 star `Score` to a 3-class label: 1-2 → negative, 3 → neutral, 4-5 → positive.

In [ ]:
df = pd.read_csv('/content/Reviews.csv')
print("Raw shape:", df.shape)

def clean_text(text):
    text = str(text)
    text = re.sub(r'<[^>]+>', ' ', text)   # strip HTML tags
    text = re.sub(r'\s+', ' ', text)        # collapse whitespace
    return text.strip()

def score_to_label(score):
    if score <= 2:
        return 'negative'
    elif score == 3:
        return 'neutral'
    else:
        return 'positive'

df['Summary'] = df['Summary'].fillna('')
df['Text_clean'] = df['Text'].apply(clean_text)
df['Summary_clean'] = df['Summary'].apply(clean_text)
df['review_text'] = df.apply(
    lambda r: f"{r['Summary_clean']}. {r['Text_clean']}" if r['Summary_clean'] else r['Text_clean'],
    axis=1
)
df['label'] = df['Score'].apply(score_to_label)

print("\nLabel distribution (before balancing):")
print(df['label'].value_counts())

## 4. Balance classes and split

The raw dataset skews heavily positive. Downsampling every class to match the minority class (`neutral`) avoids training a model that just learns to always predict "positive".

Split: 80% train / ~9% val / 10% test, stratified so each split keeps an even class balance.

In [ ]:
min_count = df['label'].value_counts().min()
print(f"Balancing each class to {min_count} rows")

balanced_df = (
    df.groupby('label', group_keys=False)[df.columns.tolist()]
    .apply(lambda x: x.sample(n=min_count, random_state=42))
    .reset_index(drop=True)
)

print("\nBalanced label counts:")
print(balanced_df['label'].value_counts())

train_val_df, test_df = train_test_split(
    balanced_df[['review_text', 'label']],
    test_size=0.10,
    stratify=balanced_df['label'],
    random_state=42
)

train_df, val_df = train_test_split(
    train_val_df,
    test_size=0.111,
    stratify=train_val_df['label'],
    random_state=42
)

print(f"\nTrain: {len(train_df)} | Val: {len(val_df)} | Test: {len(test_df)}")

## 5. Tokenize

Uses `distilroberta-base` — a distilled version of RoBERTa (~40% faster, ~half the parameters), retaining most of RoBERTa's performance while being much more practical to fine-tune on a free-tier GPU.

No static padding at tokenization time — sequences are left at their natural length, and `DataCollatorWithPadding` pads dynamically per-batch instead (only as much as each batch's longest example needs). This avoids wasting compute padding short reviews all the way to the max length.

In [ ]:
MODEL_NAME = "distilroberta-base"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

label2id = {"negative": 0, "neutral": 1, "positive": 2}
id2label = {v: k for k, v in label2id.items()}

def to_dataset(d):
    d = d.copy()
    d['label_id'] = d['label'].map(label2id)
    return Dataset.from_pandas(d[['review_text', 'label_id']].reset_index(drop=True))

def tokenize_fn(batch):
    return tokenizer(batch['review_text'], truncation=True, max_length=512)

train_ds = to_dataset(train_df).map(tokenize_fn, batched=True).rename_column('label_id', 'labels')
val_ds = to_dataset(val_df).map(tokenize_fn, batched=True).rename_column('label_id', 'labels')
test_ds = to_dataset(test_df).map(tokenize_fn, batched=True).rename_column('label_id', 'labels')

columns = ['input_ids', 'attention_mask', 'labels']
for ds in (train_ds, val_ds, test_ds):
    ds.set_format(type='torch', columns=columns)

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

print("Tokenized. Train:", len(train_ds), "Val:", len(val_ds), "Test:", len(test_ds))

## 6. Model and Trainer setup

`TrainingArguments` is built defensively — the desired config is filtered against whatever parameters the installed `transformers` version actually supports, since the API has shifted across versions (e.g. `logging_dir` and `group_by_length` were removed in some newer releases). This avoids crashing mid-run on an unsupported argument name.

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, num_labels=3, id2label=id2label, label2id=label2id,
)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        "accuracy": accuracy_score(labels, preds),
        "f1_macro": f1_score(labels, preds, average="macro"),
    }

desired_args = dict(
    output_dir="/content/checkpoints",
    num_train_epochs=5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    gradient_accumulation_steps=2,
    fp16=True,
    learning_rate=2e-5,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=3,
    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",
    logging_steps=50,
    report_to="none",
    group_by_length=True,
)

valid_params = set(inspect.signature(TrainingArguments.__init__).parameters.keys())
filtered_args = {k: v for k, v in desired_args.items() if k in valid_params}
dropped = set(desired_args.keys()) - set(filtered_args.keys())
if dropped:
    print("Dropped unsupported TrainingArguments params for this transformers version:", dropped)

training_args = TrainingArguments(**filtered_args)

trainer = Trainer(
    model=model, args=training_args,
    train_dataset=train_ds, eval_dataset=val_ds,
    compute_metrics=compute_metrics, data_collator=data_collator,
)

print(f"Ready. {MODEL_NAME}, {sum(p.numel() for p in model.parameters())} parameters.")

## 7. Train

5 epochs, checkpointing after each. `load_best_model_at_end=True` means the final in-memory model is automatically the best-performing checkpoint by validation F1-macro, regardless of which epoch that was — so later epochs overfitting doesn't cost you the better earlier result.

In [ ]:
train_result = trainer.train()
print("\nTraining complete.")
print(train_result.metrics)

## 8. Evaluate on held-out test set

This is the number that actually matters — unlike validation (which influenced checkpoint selection), the test set was never used in any training decision, so this is an unbiased estimate of real-world performance.

In [ ]:
test_results = trainer.evaluate(eval_dataset=test_ds)
print(test_results)

## 9. Export the model for serving

Finds the best checkpoint on disk (by logged validation F1-macro), copies only the files needed for *inference* (model weights, config, tokenizer — not optimizer/scheduler state, which is only needed to resume training), adds the label mapping, zips it, and triggers a download.

The resulting zip is what a serving layer (e.g. a FastAPI service) should load.

In [ ]:
checkpoint_root = "/content/checkpoints"
checkpoints = [d for d in os.listdir(checkpoint_root) if d.startswith("checkpoint-")]

best_ckpt = None
best_f1 = -1
for ckpt in checkpoints:
    state_path = os.path.join(checkpoint_root, ckpt, "trainer_state.json")
    if not os.path.exists(state_path):
        continue
    with open(state_path) as f:
        state = json.load(f)
    for entry in state.get("log_history", []):
        if "eval_f1_macro" in entry and entry["eval_f1_macro"] > best_f1:
            best_f1 = entry["eval_f1_macro"]
            best_ckpt = ckpt

print(f"Best checkpoint: {best_ckpt} (val f1_macro={best_f1:.4f})")
best_ckpt_path = os.path.join(checkpoint_root, best_ckpt)

export_path = "/content/sentiment-model-export"
os.makedirs(export_path, exist_ok=True)

keep_files = ["model.safetensors", "config.json", "tokenizer.json", "tokenizer_config.json"]
for fname in keep_files:
    src = os.path.join(best_ckpt_path, fname)
    if os.path.exists(src):
        shutil.copy(src, os.path.join(export_path, fname))
    else:
        print(f"Warning: {fname} not found in checkpoint")

with open(f"{export_path}/label_mapping.json", "w") as f:
    json.dump({"label2id": label2id, "id2label": id2label}, f)

print("Export contents:", os.listdir(export_path))

shutil.make_archive("/content/sentiment-model-export", "zip", export_path)
zip_size_mb = os.path.getsize("/content/sentiment-model-export.zip") / (1024 * 1024)
print(f"\nZip created: {zip_size_mb:.1f} MB")

In [ ]:
from google.colab import files
files.download('/content/sentiment-model-export.zip')